# ⚙️ ขั้นตอนที่ 3: Data Preparation / Preprocessing
### (การเตรียมข้อมูลและการสร้างฟีเจอร์)
---
**วัตถุประสงค์ในขั้นตอนนี้:**
1. คำนวณราคาแฟร์โดยตัดค่าน้ำ (No-Vig Probability Calculation)
2. สกัดฟีเจอร์ฟอร์มทีม (Rolling Team Statistics 5 นัดหลังสุด)
3. กำหนด Target Labels ($y$) สำหรับโจทย์ Binary Classification
4. บันทึกชุดข้อมูลพร้อมเทรนเป็นไฟล์ CSV

## 1. นำเข้าไลบรารี

In [ ]:
import os, sys
# จัดการ Working Directory ให้ชี้ไปที่โฟลเดอร์หลักของโปรเจกต์เสมอ
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
import sqlite3
import pandas as pd
import numpy as np
from fbpredict import features

print("✅ นำเข้าไลบรารีสำเร็จ")

## 2. โหลดข้อมูลดิบและจัดเรียงตามลำดับเวลา

In [ ]:
con = sqlite3.connect('data/football.db')
df = pd.read_sql('SELECT * FROM matches WHERE home_goals IS NOT NULL AND ah_fav_price IS NOT NULL', con)
con.close()

df['kickoff_at'] = pd.to_datetime(df['kickoff_at'], utc=True)
df = df.sort_values('kickoff_at').reset_index(drop=True)
print(f"โหลดข้อมูลสำเร็จ: {len(df)} แถว")

## 3. สกัดฟีเจอร์ฟอร์มทีมและสร้าง Target Labels อัตโนมัติ

In [ ]:
# ใช้ฟังก์ชันมาตรฐาน build_features ของระบบ (คำนวณฟอร์มทีม 5 นัด + ราคา No-Vig + กฎราคาเปิด)
prepared = features.build_features(df, rolling_window=5, feature_set='filtered_novig')

# คัดเฉพาะนัดที่มีป้ายกำกับเป้าหมายชัดเจน (ตัดผลเสมอราคา Push ออก)
ah_label = pd.to_numeric(prepared['ah_label'], errors='coerce')
ou_label = pd.to_numeric(prepared['ou_label'], errors='coerce')
usable_mask = ah_label.notna() & ou_label.notna()

clean_df = prepared[usable_mask].copy()
print(f"จำนวนนัดที่พร้อมเทรน (ตัดผลเสมอราคา Push ออก): {len(clean_df)} นัด")
print(f"อัตราทีมต่อชนะราคา (Fav Cover Rate): {clean_df['ah_label'].mean():.2%}")
print(f"อัตราสกอร์สูง (Over Rate):           {clean_df['ou_label'].mean():.2%}")

## 4. บันทึกชุดข้อมูลเตรียมพร้อมเทรนลงไฟล์ CSV

In [ ]:
output_path = 'data/preprocessed_matches.csv'
clean_df.to_csv(output_path, index=False, encoding='utf-8')
print(f"✅ บันทึกชุดข้อมูลเตรียมพร้อมเทรนแล้วที่: {output_path} ({len(clean_df)} นัด)")